In [13]:
import pandas as pd

In [14]:
df = pd.read_excel("fpflagc1927.xlsx", skiprows=1)
df2 = pd.read_excel("brc 27 agc 19.xlsx")

In [ ]:
# Create the new column logic
df["Rtn=Ord"] = df.apply(
    lambda x: "1" if x["RtnQty"] == x["Order"] else "0",
    axis=1
)

# Insert the column right after 'RtnQty'
rtnqty_index = df.columns.get_loc("RtnQty")
df.insert(rtnqty_index + 1, "Rtn=Ord", df.pop("Rtn=Ord"))

In [16]:
# Create the new column logic
df["Rtn+Cncl=Ord"] = df.apply(
    lambda x: "1" if x["RtnQty"] + x["Cancl"] == x["Order"] else "0",
    axis=1
)

# Insert the column right after 'Rtn=Ord'
rtnqty_index = df.columns.get_loc("Rtn=Ord")
df.insert(rtnqty_index + 1, "Rtn+Cncl=Ord", df.pop("Rtn+Cncl=Ord"))

In [17]:
# Create the new column logic
df["Ord=0"] = df.apply(
    lambda x: "1" if x["Order"] == 0 else "0",
    axis=1
)

# Insert the column right after 'Rtn=Ord'
rtnqty_index = df.columns.get_loc("Rtn+Cncl=Ord")
df.insert(rtnqty_index + 1, "Ord=0", df.pop("Ord=0"))

In [18]:
# Create the new column logic
df["Ord=Cancl+Supply+Rtn"] = df.apply(
    lambda x: "1" if x["Order"] == x["Cancl"]+x["Supply"]+x["RtnQty"] else "0",
    axis=1
)

# Insert the column right after 'Rtn=Ord'
rtnqty_index = df.columns.get_loc("Ord=0")
df.insert(rtnqty_index + 1, "Ord=Cancl+Supply+Rtn", df.pop("Ord=Cancl+Supply+Rtn"))

In [19]:
# Create "Doc Type" from PSO No (5th–6th characters)
df["Doc Type"] = (
    df["PSO No"]
    .astype(str)
    .str[4:6]   # Python is 0-based → 5th char = index 4
)

pso_index = df.columns.get_loc("Ord=Cancl+Supply+Rtn")
df.insert(pso_index + 1, "Doc Type", df.pop("Doc Type"))

In [20]:
# --- Create Release column ---
df["Release"] = df["PSO No"].astype(str).str[-1].apply(
    lambda x: "No" if x.isdigit() else "Release"
)

# Insert after "Doc Type"
doc_index = df.columns.get_loc("Doc Type")
df.insert(doc_index + 1, "Release", df.pop("Release"))

In [21]:
print(df2.columns)

Index(['branch', 'agc', 'p/n', 'FD_final', 'RC', 'MM06', 'MM12', 'Calls12'], dtype='object')


In [22]:
# --- Normalize part numbers (CRITICAL) ---
df["Partno"] = df["Partno"].astype(str).str.strip().str.upper()
df2["p/n"] = df2["p/n"].astype(str).str.strip().str.upper()

# --- Select only needed columns from df2 ---
df2_merge = df2[["p/n", "FD_final", "RC", "MM06","MM12","Calls12"]].drop_duplicates()

# --- Merge ---
df = df.merge(
    df2_merge,
    left_on="Partno",
    right_on="p/n",
    how="left"
)

# --- Drop duplicate key column ---
df.drop(columns=["p/n"], inplace=True)

In [23]:
df.to_excel(f"fpfldet agc19 brc27.xlsx", index=False)